# Session 9a · Environment and your first LLM call
## One repo, one Python, one config file, one rule — and `hello.py` rebuilt cell by cell

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why a careful setup matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 10. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps, so you always know where you are:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

**Before you open this notebook** you should have done the terminal steps in the README, *Session 2 — set up your machine* (steps 1 to 6). If you have not, do them first: this notebook checks them, it does not replace them. Open the notebook from the course folder with:

```powershell
cd ~\Documents\genai-agents-course
uv run jupyter lab module01/s09a_first_call.ipynb
```

**No key yet?** You can still run every cell. The cells that call the model will print `skipped (...)` instead of crashing, and the "practice without a key" cells show you exactly what a real answer looks like.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
A bank's digital team builds a small assistant that answers customer questions about cards and loans. The demo for the head of retail banking is on Monday. On Friday it works perfectly on the developer's laptop. Then three things happen.

1. **"It works on my laptop."** On Monday morning a colleague runs the same notebook on the meeting-room laptop. It crashes with an error nobody has seen. That laptop has a different Python and an older version of one package. Forty minutes of the meeting go on fixing the laptop instead of showing the assistant.
2. **The key in the code.** To save time, the developer had pasted the API key straight into the notebook. Over the weekend he pushed the notebook to a public GitHub repository so a friend could look at it. Automated scanners look for keys in public code all the time; someone finds this one and uses it. The bank's account hits its limit and the provider blocks the key. The security team now has an incident report to write.
3. **The hard-wired provider.** The provider's address and the model name were also written inside the code, in four different files. When the security team asks, *"Can we run this on a model inside our own network for the pilot with real customer data?"*, the answer is "we need to change and re-test four files", not "we change one setting".

None of these is about AI. **They are about setup.** The model was fine every time. Today you set up your laptop so that none of these three things can happen to you, and then you make your first call to a language model, step by step, so that you understand every line.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Give the project its **own Python 3.12 and pinned packages** with `uv`, and start every command with `uv run` | "It works on my laptop" crashes | One toolbox per project |
| Keep the key and settings in **`.env`**, which Git ignores | A key on the internet within minutes | Your locker, not the notice board |
| Read **three settings** (`BASE_URL`, `API_KEY`, `MODEL`) instead of writing them in code | Changing code to change provider | Change the setting, not the code |
| Build the **messages list** and read the **reply** and its **token usage** | Calls you cannot explain or price | Know what you sent and what it cost |

### Where this is used in industry
- **Banking and insurance:** secrets never live in code. On laptops they live in a git-ignored file like `.env`; in production they live in a secrets manager (for example Azure Key Vault, AWS Secrets Manager or HashiCorp Vault) and reach the program as environment variables. The code reads them the same way in both places: `os.getenv(...)`. You meet the production side in Module 12.
- **Every regulated industry (BFS, healthcare, telecom):** auditors and incident teams ask *"exactly which versions were running?"*. A lock file (`uv.lock` here; other tools have their own) answers that, and lets anyone rebuild the same environment.
- **Provider choice:** many hosted providers and local servers (Groq and Ollama among them; Gemini in Session 11) accept the same **OpenAI-compatible** request. Teams use this to switch between a hosted model and a model inside their own network by changing configuration only. Hospitals and banks do exactly this for data that must not leave the building (Session 12).
- **Chat products:** every chat assistant you have used **streams** its answer, so the first words appear quickly instead of after a long wait. And every production team logs **token usage** per call, because tokens are the bill (Sessions 19 and 53).

### Why interviewers care
These questions come up in GenAI-developer and AI-engineer interviews, often in the first fifteen minutes:
- *"Where do you keep API keys, and what do you do if one leaks?"*
- *"How do you make a Python project reproducible on another machine?"*
- *"What does 'OpenAI-compatible API' mean? How would you switch from a hosted model to a local one?"*
- *"What are the system and user roles in a chat request?"*
- *"Why would you stream a response, and how do you still get the token count?"*

By the end of today you can answer each one with a definition, an example from your own lab, and one honest limit. Part C has model answers.

<details><summary><b>▸ Read more: why spend a whole session on setup?</b></summary>

In a course of thirty people on thirty laptops, the thing that eats the most time is not a hard concept. It is a laptop that behaves differently from its neighbour. The same is true in companies: a large share of "the model is broken" tickets turn out to be a wrong key, a wrong environment or a wrong setting. Engineers who can find those problems in two minutes, by reading the error and checking three settings, are the ones teams trust with production. Today's habits (`uv run`, `.env`, three settings, read the error) are the ones you will use in every session until the capstone.
</details>

## A2. Where this session fits in the course

```
 S1 / Git webinar  Git & GitHub ───► clone, pull, commit, push; .gitignore lists .env
  │
 S9a Environment + first call ─────► TODAY: uv, Python 3.12, uv sync, .env, the three settings,
  │                                  hello.py rebuilt cell by cell, tokens read from the reply
 S9b How LLMs work ────────────────► tokens in depth: why the provider counts more tokens than
  │                                  you typed, what a call costs, the context window
 S10 The model landscape ──────────► choosing a model; why MODEL is pinned for the whole batch
  │
 S11 Multi-provider calls ─────────► today's client, three providers (Groq, Gemini, Ollama);
  │                                  temperature, max_tokens, roles in more depth
 S12 Local and open models ────────► Ollama properly: the fallback you install tonight
  │
 S13 Images + structured output ───► the same messages list, with an image inside
 S14 Prompt anatomy ───────────────► your own work repository; the system message as a design
```

This session is the course's **Session 2** ("Environment and your first LLM call"), given at the start of Module 1. If you are starting the course here, the README tells you to do only step 8 of Session 1 (clone the course repository) and then all of Session 2's setup. The Git & GitHub webinar covers the rest of Session 1 before Session 14.

**What you need from earlier sessions:** Git installed and the course repository cloned into `Documents` (README Session 1, step 8). Nothing else. Everything else is set up today.

## A3. The words you need today

Read these once now. Each term appears in bold the first time it is used in Part B.

---
**1. Repository (repo): the course repo and your work repo**
- **Definition:** a project folder that Git tracks, with its full history. This course uses **two**, side by side in `Documents`: the **course repository** `genai-agents-course` (the instructor's; you only `git pull` it) and your **work repository** `genai-course-work` (yours; you `git add`, `commit` and `push` your submissions to it, from Session 14).
- **Analogy:** a folder with a memory. The course repo is the textbook you photocopy and collect new pages for; the work repo is your own exercise book that you hand in.
- **Example:** `git pull` in `genai-agents-course` brings today's new notebook to your laptop.
- **Often confused with:** each other. Never put one inside the other, and never push your own work to the course repo.
- **Comes back in:** S14 (your work repo), every session (`git pull`).

---
**2. uv**
- **Definition:** a tool that installs Python versions, creates the project's environment and installs the project's packages.
- **Analogy:** the toolbox manager: it fetches the right tools and puts them in this project's box.
- **Example:** `uv python install 3.12`, `uv sync`, `uv run python hello.py`.
- **Often confused with:** pip or conda. They also install packages; uv does Python, environment and pinned packages in one fast tool, and it never touches your Anaconda.
- **Comes back in:** every session.

---
**3. Virtual environment (`.venv`)**
- **Definition:** a folder inside the project that holds the project's own Python and its own packages.
- **Analogy:** the project's toolbox. Other projects have their own boxes; nobody borrows tools from yours.
- **Example:** `uv sync` creates `genai-agents-course\.venv`.
- **Often confused with:** something you edit or upload. You never edit it and never commit it; uv can delete and rebuild it any time.
- **Comes back in:** every session.

---
**4. `pyproject.toml` and the lock file `uv.lock`**
- **Definition:** `pyproject.toml` is the list of packages the project needs (with allowed version ranges); `uv.lock` pins the **exact** version of every package, including the packages those packages need.
- **Analogy:** the shopping list and the exact receipt.
- **Example:** `pyproject.toml` says `openai>=1.60,<2`; `uv.lock` says exactly which 1.x version everyone gets.
- **Often confused with:** `requirements.txt`. Similar idea; the lock file is more complete and is created by the tool, not typed by hand.
- **Comes back in:** S9b, and whenever packages are added.

---
**5. `uv sync` and `uv run`**
- **Definition:** `uv sync` makes `.venv` match the lock file exactly. `uv run <command>` runs a command with the project's Python and packages (and syncs first if needed).
- **Analogy:** `uv sync` packs the toolbox from the receipt; `uv run` says "use the tools from *this* box".
- **Example:** `uv run pytest tests/test_setup.py`.
- **Often confused with:** typing plain `python`. Plain `python` runs whichever Python Windows finds first, which differs from laptop to laptop.
- **Comes back in:** every session. *The course rule: every Python command starts with `uv run`.*

---
**6. Environment variable**
- **Definition:** a named setting (name = value) that a running program can read, for example with `os.getenv("MODEL")`.
- **Analogy:** a note stuck on the program's door before it starts work.
- **Example:** after `load_dotenv`, `os.getenv("BASE_URL")` returns `https://api.groq.com/openai/v1`.
- **Often confused with:** a Python variable. A Python variable lives in your code; an environment variable lives outside it, in the process that runs your code.
- **Comes back in:** every lab; Module 12 (deployment).

---
**7. `.env` and `.env.example`**
- **Definition:** `.env` is a small private text file of settings (`NAME=value` lines) that `load_dotenv` turns into environment variables. `.env.example` is the public template with placeholders.
- **Analogy:** `.env.example` is the blank form pinned on the notice board; `.env` is your locker.
- **Example:** `copy .env.example .env`, then paste your key on the `API_KEY=` line.
- **Often confused with:** each other. Editing `.env.example` by mistake is one of the classic errors.
- **Comes back in:** every lab; S11 (Gemini key), S16 (`WORK_REPO`).

---
**8. `.gitignore`**
- **Definition:** a list of files and folders Git must never track. The course repo lists `.env` (and `.venv`) in it.
- **Analogy:** "do not photocopy these".
- **Example:** `git add .` skips `.env`, so `git push` cannot upload your key.
- **Often confused with:** deleting or hiding. The file is still on your laptop; Git simply ignores it.
- **Comes back in:** S14 (your work repo gets its own `.gitignore`), every submission.

---
**9. API and API key**
- **Definition:** an API is a way for one program to ask another program, over the internet, to do something. An **API key** is your secret credential for that service.
- **Analogy:** the key is your ATM PIN. The PIN is not the money, but it lets somebody else spend it.
- **Example:** your Groq key, created at console.groq.com, stored only in `.env`.
- **Often confused with:** a password you can reuse. If it leaks, you delete it in the console and create a new one (two minutes), you do not "change it a little".
- **Comes back in:** S11 (a second key), S55–S57 (security).

---
**10. `BASE_URL`, `API_KEY`, `MODEL`: the three settings**
- **Definition:** **where** the request goes, **who** is asking, and **which model** answers.
- **Analogy:** the address on the envelope, your signature, and the name of the expert you want to answer.
- **Example:** `https://api.groq.com/openai/v1` · your key · `openai/gpt-oss-20b`.
- **Often confused with:** code. None of the three is in the code; that is the point.
- **Comes back in:** S10 (why `MODEL` is pinned), S11 (switching providers), S12 (Ollama).

---
**11. Provider and OpenAI-compatible API**
- **Definition:** a **provider** is the company or server that runs the model (Groq, Ollama on your laptop, Gemini). **OpenAI-compatible** means it accepts the same request format as OpenAI's Chat Completions API, so one client library talks to all of them.
- **Analogy:** different banks' ATMs that all accept the same card.
- **Example:** the same `hello.py` talks to Groq or Ollama; only the three settings change.
- **Often confused with:** "uses OpenAI's models". It means *same format*, not *same company or model*.
- **Comes back in:** S11, S12.

---
**12. Client (client library)**
- **Definition:** a Python object, from the `openai` package, that knows how to send requests to a provider and turn the replies into Python objects.
- **Analogy:** the phone you dial with. Making the phone does not call anyone.
- **Example:** `client = OpenAI(base_url=BASE_URL, api_key=API_KEY)`.
- **Comes back in:** S11, and every lab after.

---
**13. Messages and roles (system, user)**
- **Definition:** the request is a **list of messages**. Each has a **role** and **content**. **system** = who the model is and how it behaves; **user** = what was asked. (The model's reply has the role **assistant**.)
- **Analogy:** the system message is the job description you give a new employee; the user message is the customer at the counter.
- **Example:** system: *"You are a concise assistant for a retail bank."* user: *"Say hello and tell me one thing AI agents can do."*
- **Often confused with:** one big string. The roles matter: the model is trained to treat them differently.
- **Comes back in:** S11, S14 (anatomy of a prompt), S19 (conversation history).

---
**14. Streaming and chunks**
- **Definition:** with `stream=True` the reply arrives in many small pieces (**chunks**) while the model writes it, instead of all at once at the end.
- **Analogy:** watching a live cricket score update ball by ball, instead of reading the result in tomorrow's paper.
- **Example:** hello.py prints each chunk's text with `end=""` so the words join up on one line.
- **Often confused with:** a faster model. The total time is about the same; you just see the first words much sooner.
- **Comes back in:** S11, S22 (serving), S53 (time to first token).

---
**15. Tokens and usage (prompt, completion, total)**
- **Definition:** a **token** is the unit a model reads, writes and bills by (roughly three-quarters of an English word). **Usage** is the count the provider reports after a call: **prompt** tokens (what you sent, plus some the provider adds), **completion** tokens (what the model produced), **total**.
- **Analogy:** the bill, in tokens.
- **Example:** an illustration from the deck: `prompt=96 completion=105 total=201`.
- **Often confused with:** words or characters. Session 9b shows exactly how text becomes tokens.
- **Comes back in:** S9b, S15 (reasoning tokens), S19 (token budgets), S53 (cost per run).

---
**16. Checkpoint test (pytest)**
- **Definition:** a small program that checks your work and reports `passed`, `failed` or `skipped`.
- **Analogy:** the examiner. Green, or you are not finished.
- **Example:** `uv run pytest tests/test_setup.py` → `3 passed`.
- **Often confused with:** "it ran without errors". A skipped test is not a pass.
- **Comes back in:** every session; S51 (evaluation-driven development).

## A4. How this session is built

### The big picture

```
┌──── COURSE REPOSITORY  genai-agents-course  (you pull it; you do not edit it) ───────────────────────────────┐
│                                                                                                              │
│  pyproject.toml + uv.lock     the shopping list and the exact receipt: Python 3.12, openai, python-dotenv,   │
│                               pytest, jupyter ...   uv sync reads these two files                            │
│  .env.example                 the public template: BASE_URL, API_KEY (placeholder), MODEL (pinned)           │
│  .gitignore                   tells Git never to track .env or .venv                                         │
│                                                                                                              │
│  hello.py                     Session 2's first call: load settings, client, messages, streamed reply        │
│  list_models.py               which models can my key use?                                                   │
│  tests/test_setup.py          THE CHECKPOINT: Python 3.12? real key? does the model answer?                  │
│  tests/test_s09a.py           checks that this notebook file is intact (offline)                             │
│                                                                                                              │
│  module01/s09a_first_call.ipynb   ◄── YOU ARE HERE: hello.py rebuilt cell by cell, with explanations         │
│                                                                                                              │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                                   │ uv run  ·  load_dotenv reads .env  ▼
┌──── YOUR MACHINE  (yours; never committed to the course repo) ───────────────────────────────────────────────┐
│  .venv\              made by uv sync: the project's Python 3.12 + the exact packages (never edit it)         │
│  .env                your copy of .env.example with YOUR key (git-ignored, never uploaded)                   │
│  uv's Python 3.12    installed once by uv python install 3.12; your Anaconda/python.org stays untouched      │
│  Ollama (optional)   a local model server at http://localhost:11434 (install at home tonight)                │
│  genai-course-work\  your work repository, next to the course repo (needed from Session 14)                  │
└──────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                                   │ HTTPS request to BASE_URL  ▼
                        Groq API (default)   or   Ollama on your laptop (fallback)
```

**Read it top to bottom.** The course repo gives you the code and the template. Your machine adds the three private things the code needs: the environment (`.venv`), the settings (`.env`) and, optionally, a local model. The notebook joins them and sends one request to whichever provider `BASE_URL` names.

### What happens to one prompt

```
 .env  (BASE_URL, API_KEY, MODEL)
   │
   ▼  load_dotenv(..., override=True)            Block 1: settings become environment variables
 os.getenv("BASE_URL"), os.getenv("API_KEY"), os.getenv("MODEL")
   │
   ▼  OpenAI(base_url=..., api_key=...)          Block 2: the client (nothing is sent yet)
 client
   │
   ▼  messages = [system, user]                  Block 3: what you want to say
 client.chat.completions.create(model=MODEL, messages=messages, stream=True,
                                stream_options={"include_usage": True})
   │
   ▼  HTTPS POST  BASE_URL + "/chat/completions"   (key in the Authorization header)
 the model, at Groq (or Ollama)
   │
   ▼  chunks arrive one by one                   Block 4: print each piece of text as it comes
 "Hello" "! One" " thing" ...  and a last chunk with usage
   │
   ▼
 prompt=…  completion=…  total=…                 the bill, in tokens
```

### The functions you will call, and what each does

There are no course helper files today: you use the libraries directly, exactly like `hello.py`.

| Function | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `find_dotenv(usecwd=True)` | python-dotenv | nothing (it searches from the current folder upwards) | the path of `.env`, or `""` if not found | 2 |
| `load_dotenv(path, override=True)` | python-dotenv | the path of `.env` | `True`/`False`; the settings are now environment variables | 2 |
| `dotenv_values(stream=...)` | python-dotenv | `.env`-style text | a dictionary of the settings, without touching the environment | 2 (under the hood) |
| `os.getenv(name)` | Python (`os`) | a setting's name | its value, or `None` | 2 |
| `mask(secret)` | this notebook | a key | a safe version to print: start, end and length only | 2 onwards |
| `OpenAI(base_url, api_key)` | openai | where and who | a client (sends nothing yet) | 3 |
| `explain(error)` | this notebook | an error from a call | prints the matching fix from the README | 5 onwards |
| `client.chat.completions.create(...)` | openai | model + messages (+ `stream`) | a response object, or a stream of chunks | 5, 6, 7 |
| `client.models.list()` | openai | nothing | the models your key can use (like `list_models.py`) | 8 |

### The order of the sections

| # | Section | Calls the model? | Minutes |
|---|---|---|---|
| 1 | Your setup, and a check that it worked | no | 6 |
| 2 | Block 1 · Load your settings from `.env` | no | 10 |
| 3 | Block 2 · Make the client | no | 4 |
| 4 | Block 3 · The messages: what you actually send | no | 6 |
| 5 | Block 4a · The first call, all at once | yes, 1 call | 7 |
| 6 | Block 4b · The same call, streamed (exactly like `hello.py`) | yes, 1 call | 7 |
| 7 | Read the bill: change the question, compare the tokens | yes, 1 call per run | 6 |
| 8 | One code, two providers: Groq or Ollama | no tokens (1 request for the model list) | 5 |
| 9 | The checkpoint | yes, 1 tiny call (inside the test) | 4 |
| 10 | Reflection | no | 3 |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. Explain why a project needs its own Python and pinned packages, and what `uv run` guarantees.
2. Create the environment from the course repository with one command (`uv sync`) and verify it.
3. Keep credentials in `.env`, explain why it is git-ignored, and read the three settings from code.
4. Build a messages list with a system and a user message, and send it with an OpenAI-compatible client.
5. Read a reply both ways, all at once and streamed, and read its prompt, completion and total tokens.
6. Switch provider (Groq ↔ Ollama) by changing `.env` only, and diagnose the four classic errors from their messages.

**The checkpoint (in the terminal, in the course folder):**

```powershell
uv run pytest tests/test_setup.py
```

It must say **`3 passed`**: Python 3.12, a real key in `.env`, and the model answers. Show it to the lab assistant. Section 9 runs the same test from the notebook if you have a key, but the terminal run is the one that counts. *A failed or skipped test is not a pass.*

There is nothing to submit to a repository today. Your own work repository starts in Session 14.

**Budget:** each call today uses a few hundred tokens, so the whole notebook uses a few thousand at most: very small compared with the free key's daily limit (the README quotes 200,000 tokens a day). The free tier needs no card; when you hit a limit you wait, you are not charged.

---
# Part B · Hands-on

## 1. Your setup, and a check that it worked

### What and why
In the terminal you ran six steps from the README: installed **uv**, installed Python 3.12 through uv, cloned the **course repository**, ran **`uv sync`**, and copied `.env.example` to `.env`. This notebook starts by checking the result, because every later section depends on it. If something is wrong, you want to know *now*, with a clear message, not in section 6 with a confusing one.

Here is what each terminal step really did:

| Step | Command | What it really did |
|---|---|---|
| uv | `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 \| iex"` then `uv self version` | downloaded the uv program into your user folder (`~\.local\bin`) and added it to `PATH`. Only a **new** terminal sees a new `PATH`, which is why you had to close and reopen it |
| Python | `uv python install 3.12` then `uv python list` | downloaded a separate Python 3.12 into uv's own folder. It never touches the Python or Anaconda you already had |
| Clone | `cd ~\Documents` · `git clone https://github.com/byruajit/genai-agents-course.git` | made a full copy of the course repository in `Documents\genai-agents-course` |
| Packages | `uv sync` | read `pyproject.toml` and `uv.lock`, created `.venv` inside the course folder, and installed exactly the pinned packages |
| Settings | `copy .env.example .env`, then `notepad .env` | made your private copy of the settings, where your key goes |

**Two repositories, two jobs.** The course repository is the instructor's: you only ever `git pull` it, at the start of every session. Your **work repository** (`genai-course-work`, next to it in `Documents`) is yours: from Session 14 you commit and push your submissions there. Never put one inside the other.

### The idea: three Pythons on one laptop
```
 ONE LAPTOP, WITHOUT uv                         ONE LAPTOP, WITH uv
 ──────────────────────────────────             ─────────────────────────────────────────────
  Anaconda Python 3.11    ◄─┐                    Anaconda, python.org, Store: still there,
  python.org Python 3.13  ◄─┼─ which one runs     never used for this course
  Microsoft Store "python"◄─┘  when you type
                               "python"?         genai-agents-course\
  answer: whichever Windows                         .venv\  ◄── Python 3.12 + the exact packages
  finds first. Different on                         uv run ──► always uses THIS one
  every laptop.
```
**Analogy — the toolbox.** Every project gets its own toolbox (`.venv`), packed from the same receipt (`uv.lock`). `uv run` means "use the tools from this box". Your neighbour's box has exactly the same tools, down to the version number, so "it works on my laptop" stops being a problem.

If your prompt shows `(base)`, that is Anaconda switching itself on. Ignore it; do not deactivate it. As long as you type `uv run`, it does not matter.

### What the next cell does
1. **Imports.** `sys` tells us about the running Python. `Path` (from **`pathlib`**) is Python's way of handling file paths; `Path.cwd()` is the folder the notebook is running in. `version` reads an installed package's version.
2. **Finds the course folder.** The `for` loop walks from the current folder up through its **parents** (`[Path.cwd(), *Path.cwd().parents]` is a list: the current folder, then each parent; the `*` unpacks the parents into the list). The first folder that contains `pyproject.toml` is the course folder; `break` stops the loop there. This works whether Jupyter started in the course folder or in `module01`.
3. **Prints the Python version and the Python program's path.** `sys.version.split()[0]` keeps just the number, like `3.12.x`.
4. **Checks `.venv`.** `sys.prefix` is the folder of the running Python. If it is the course folder's `.venv`, you opened Jupyter the right way.
5. Prints a warning if the version is not 3.12.
6. **Checks three packages** with **`try` / `except`**: Python *tries* the indented lines; if a `PackageNotFoundError` happens, it runs the `except` lines instead of crashing. Here that means a clear "run `uv sync`" message.
7. The **f-string** `f"{package:<14} : ..."` puts the value inside the text; `:<14` pads it to 14 characters so the colons line up.
8. Warns if you are working inside `C:\Windows\system32` (where an administrator PowerShell opens by default, and the wrong place for anything of ours).

In [4]:
# Section 1 · Is this the project's Python?   (no key, no internet needed)
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

REPO = Path.cwd()                                        # fallback: the folder Jupyter started in
for folder in [Path.cwd(), *Path.cwd().parents]:         # this folder, then each parent folder
    if (folder / "pyproject.toml").exists():             # the course folder is the one with pyproject.toml
        REPO = folder
        break

print("Python version :", sys.version.split()[0])
print("Python program :", sys.executable)
print("course folder  :", REPO)
in_project_venv = Path(sys.prefix).resolve() == (REPO / ".venv").resolve()
print("project .venv  :", "yes" if in_project_venv else "no  <-- start Jupyter with 'uv run jupyter lab' in the course folder")
if sys.version_info[:2] != (3, 12):
    print("WARNING: the course uses Python 3.12. You are probably not running through 'uv run'.")

for package in ("openai", "python-dotenv", "pytest"):
    try:
        print(f"{package:<14} : {version(package)}")
    except PackageNotFoundError:
        print(f"{package:<14} : NOT INSTALLED  <-- run 'uv sync' in the course folder, then restart the kernel")

if "system32" in str(Path.cwd()).lower():
    print("WARNING: you are inside C:\\Windows\\system32. Close this, run 'cd ~\\Documents\\genai-agents-course', start again.")

Python version : 3.12.14
Python program : d:\genai-agents-course\.venv\Scripts\python.exe
course folder  : d:\genai-agents-course
project .venv  : yes
openai         : 1.109.1
python-dotenv  : 1.2.3
pytest         : 8.4.2


### What you should see
Something like this (an illustration, not a real run; your paths and patch numbers will differ):
```
Python version : 3.12.x
Python program : C:\Users\you\Documents\genai-agents-course\.venv\Scripts\python.exe
course folder  : C:\Users\you\Documents\genai-agents-course
project .venv  : yes
openai         : 1.x.x
python-dotenv  : 1.x.x
pytest         : 8.x.x
```
**How to read it:** the important words are `3.12`, a program path that contains `genai-agents-course\.venv`, and `project .venv : yes`. Everybody who used `uv sync` should see the same package versions, because they all came from the same `uv.lock`. Compare with your neighbour: that is the whole point of the lock file.

| If you see... | It means... | Do this |
|---|---|---|
| `Python version : 3.11` or `3.13`, and a path with `anaconda3` | Jupyter was started without `uv run` (or VS Code picked the wrong kernel) | Close it. In the course folder: `uv run jupyter lab module01/s09a_first_call.ipynb`. In VS Code: *Select Kernel* → the `.venv` of the course folder |
| `NOT INSTALLED` | `uv sync` was not run, or failed halfway | `uv sync` in the course folder, then **Kernel → Restart** |
| `project .venv : no` but version 3.12 | a different 3.12 is running | same fix as the first row; it matters for the checkpoint |
| the `system32` warning | the terminal was in the wrong folder | `cd ~\Documents\genai-agents-course` and start again |

### Check yourself
*Question 1 — Your neighbour typed `python hello.py` instead of `uv run python hello.py`, and it failed with `No module named 'dotenv'`. Using the "three Pythons" picture, explain why.*

Answer: The command may be using a different Python environment instead of the project’s virtual environment. The course uses uv to manage its Python version and packages.

How to fix it:

("uv sync")

("uv run jupyter lab module01/s09a_first_call.ipynb")


Use uv run to run commands inside the project environment.

> **Common mistake:** "fixing" a missing package with `pip install` into whatever Python is first on the PATH. That installs into the wrong toolbox. The course fix is always `uv sync`, then `uv run`.

## 2. Block 1 · Load your settings from `.env`

### What and why
`hello.py` starts with two lines that look small and matter a lot:

```python
load_dotenv(override=True)   # .env wins over anything an editor put in the environment
client = OpenAI(base_url=os.getenv("BASE_URL"), api_key=os.getenv("API_KEY"))
```

The code never contains the key, the provider's address or the model name. It reads all three from **`.env`**, a private file that only exists on your laptop. There are two reasons:

1. **Safety.** Code gets shared: pushed to GitHub, pasted in chat, shown on a projector. If the key is in the code, the key goes wherever the code goes. `.env` is listed in **`.gitignore`**, so `git add` skips it and it cannot be pushed by accident.
2. **Flexibility.** To use a different provider or model you change the setting, not the code. The same `hello.py` runs against Groq today and against Ollama tonight.

### The idea: the locker and the notice board
```
 .env.example  (in the repo; everyone has it)          .env  (only on YOUR laptop; Git ignores it)
 ───────────────────────────────────────────           ────────────────────────────────────────────
 BASE_URL=https://api.groq.com/openai/v1               BASE_URL=https://api.groq.com/openai/v1
 API_KEY=paste_your_groq_key_here       ── copy ──►    API_KEY=<your own key, no quotes, no spaces>
 MODEL=openai/gpt-oss-20b                              MODEL=openai/gpt-oss-20b    (do not change)
                                                                  │
                                         load_dotenv  ────────────┘  turns each line into an
                                                                     environment variable
                                         os.getenv("API_KEY")  ◄──── the code reads it by name
```
**Analogy.** `.env.example` is the blank form pinned on the notice board: everybody can see it, and it has a placeholder where the key goes. `.env` is your locker: same shape, your own contents, and only you have it. Your key is like your ATM PIN: it goes in the locker and nowhere else.

**The three settings:** `BASE_URL` = *where* the request goes; `API_KEY` = *who* is asking; `MODEL` = *which* model answers. `MODEL` is pinned for the whole batch: if a model is retired, the instructor changes it for everyone.

**Why `override=True`?** An editor (VS Code, for example) can put its own copy of these variables into the environment before your code starts, sometimes an old value. `override=True` means *the values in `.env` win*. One earlier version of `.env.example` had a comment after the `MODEL` value, and some editors read the whole line, comment included, as the model name; with `override=True` the clean value from `.env` replaces it.

### What the next cell does
1. **`find_dotenv(usecwd=True)`** looks for a file called `.env` in the current folder, then in each parent folder, and returns its path (or `""` if there is none). In `hello.py` plain `load_dotenv()` finds it by looking next to the script file; a notebook is not a script file, so we tell it to start from the current folder (`usecwd=True`). Either way it finds the `.env` in the course folder.
2. **`load_dotenv(ENV_PATH, override=True)`** reads that file and sets each `NAME=value` line as an environment variable. If the path is `""`, it simply loads nothing.
3. **`os.getenv("BASE_URL")`** and the next two lines read the three settings. `os.getenv` returns `None` if a name does not exist (no crash).
4. **`def mask(secret):`** defines a small function that hides a key for printing: the first 4 and last 4 characters plus the length, or stars for short values. You never print a full key, not even on your own screen: screens get shared and screenshots get posted.
5. **`placeholder = bool(API_KEY) and "paste_your" in API_KEY`** is `True` if you forgot to replace the placeholder. `and` stops early: if there is no key at all, it does not try to search inside `None`.
6. **`ready = ...`** is `True` only if all three settings exist and the key is not the placeholder. Every model cell later checks `ready` and prints `skipped (...)` if it is `False`.
7. **`ENV_PATH or "NOT FOUND ..."`** prints the path if there is one, otherwise the message (`or` returns the first value that is not empty). The line for the key uses a **conditional expression**, `A if condition else B`: one line that picks between two values.

In [5]:
# Section 2 · Block 1 of hello.py: load your settings   (runs without a key)
import os
from dotenv import load_dotenv, find_dotenv

ENV_PATH = find_dotenv(usecwd=True)           # look for .env here, then in each parent folder
load_dotenv(ENV_PATH, override=True)          # .env wins over anything an editor put in the environment

BASE_URL = os.getenv("BASE_URL")              # WHERE the request goes
API_KEY = os.getenv("API_KEY")                # WHO is asking (your secret: never print it in full)
MODEL = os.getenv("MODEL")                    # WHICH model answers (pinned for the batch)


def mask(secret):
    # Show only the start and the end of a secret, never the whole thing.
    if not secret:
        return "(missing)"
    if len(secret) < 12:
        return "*" * len(secret)
    return f"{secret[:4]}...{secret[-4:]} ({len(secret)} characters)"


placeholder = bool(API_KEY) and "paste_your" in API_KEY
ready = bool(BASE_URL and API_KEY and MODEL) and not placeholder    # every model cell checks this

print(".env found at :", ENV_PATH or "NOT FOUND  <-- in the course folder: copy .env.example .env")
print("BASE_URL      :", BASE_URL or "(missing)")
print("API_KEY       :", "still the placeholder  <-- paste your own key" if placeholder else mask(API_KEY))
print("MODEL         :", MODEL or "(missing)")
print("ready         :", ready)
if not ready:
    print("\nNo usable key: the cells that call the model will print 'skipped'. Everything else still runs.")

.env found at : d:\genai-agents-course\.env
BASE_URL      : https://api.groq.com/openai/v1
API_KEY       : gsk_...pxVa (56 characters)
MODEL         : openai/gpt-oss-20b
ready         : True


### What you should see
With a key (an illustration, not a real run):
```
.env found at : C:\Users\you\Documents\genai-agents-course\.env
BASE_URL      : https://api.groq.com/openai/v1
API_KEY       : gsk_...a1b2 (56 characters)
MODEL         : openai/gpt-oss-20b
ready         : True
```
Without a key, `ready : False` and the "No usable key" line. That is fine for now: keep going, and fix `.env` before section 5.

| If you see... | It means... | Do this |
|---|---|---|
| `.env found at : NOT FOUND` | you have no `.env`, or it is in the wrong folder, or Windows saved it as `.env.txt` | in the course folder: `copy .env.example .env`. In File Explorer turn on *View → File name extensions* to spot `.env.txt` |
| `still the placeholder` | you edited `.env.example`, or did not save `.env` | open `.env` (check the tab name in Notepad or VS Code), paste the key, save |
| `API_KEY : (missing)` | the line is misspelled, e.g. `API KEY=` or `APIKEY=` | the name must be exactly `API_KEY` |
| the key length looks wrong | the copy from the console was incomplete, or included extra characters | create a new key in the console and paste it again |
| you edited `.env` but nothing changed | Python already read the old values | **Kernel → Restart**, then run from the top |

### Check yourself
*Question 2 — Name the three settings and say, in one word each, what they answer (where? who? which?). Then: why is it safe to push `hello.py` to GitHub, but not `.env`?*
Answer:

- BASE_URL: tells the program where to send the API request.

- API_KEY: identifies the credentials used to access the provider.

- MODEL: specifies which model to use.

- .env: stores configuration and secret values outside the source code. Git should ignore this file so the API key is not accidentally uploaded.

The source code can be committed to GitHub, but a file containing a real API key must not be committed.

> **Common mistake:** printing the key "just to check" with `print(API_KEY)`, and then sharing a screenshot of the notebook. Use `mask(API_KEY)`. If a full key was ever visible in a screenshot, chat or commit, delete it in the console and create a new one.

### Under the hood 1: how the text in `.env` becomes settings
`load_dotenv` is not magic: it reads text lines of the form `NAME=value` and turns them into settings. The function `dotenv_values` does the same reading, but returns a dictionary instead of changing the environment. That lets us experiment safely with **made-up values** in a string, without touching your real `.env`.

The course rules for `.env` are strict: no quotes, no spaces, no comment after a value. This cell shows which mistakes python-dotenv forgives, and which ones it does not. (Other tools, such as some editors, read `.env` more simply, so the strict rules protect you everywhere.)

**What the next cell does:**
1. `sample_env` is a multi-line string (written between triple quotes) that looks like a `.env` file, with good lines and typical mistakes. All values are fake.
2. **`io.StringIO(sample_env)`** makes the string behave like an open file, so `dotenv_values(stream=...)` can read it.
3. The `for name, value in settings.items():` loop prints each setting. **`.items()`** gives pairs (name, value) and the loop unpacks each pair into two variables. `{value!r}` prints the value with quotes around it, so you can see hidden spaces or curly quotes.
4. The `assert` lines check what we claim: an `assert` does nothing if the statement is true, and stops with an error if it is false.

In [6]:
# Under the hood 1 · how .env text becomes settings   (fake values only; your real .env is not touched)
import io
from dotenv import dotenv_values

sample_env = """\
# a line that starts with # is a comment and is ignored
BASE_URL=https://api.groq.com/openai/v1
MODEL=openai/gpt-oss-20b
API_KEY=demo_key_one
QUOTED="demo_key_two"
SPACES =   demo_key_three
SMART_QUOTES=“demo_key_four”
COMMENT_AFTER=openai/gpt-oss-20b # pinned for the batch
HASH_NO_SPACE=openai/gpt-oss-20b# pinned
"""
settings = dotenv_values(stream=io.StringIO(sample_env))      # the same parser load_dotenv uses
for name, value in settings.items():
    print(f"{name:<14} -> {value!r}")

assert settings["QUOTED"] == "demo_key_two"                   # straight quotes: removed (forgiven)
assert settings["SPACES"] == "demo_key_three"                 # spaces round '=': removed (forgiven)
assert settings["COMMENT_AFTER"] == "openai/gpt-oss-20b"      # ' # comment': removed (forgiven)
assert settings["SMART_QUOTES"] != "demo_key_four"            # curly quotes from Word/WhatsApp: KEPT -> wrong key
assert settings["HASH_NO_SPACE"] != "openai/gpt-oss-20b"      # '#' without a space: KEPT -> wrong model name
print("\nAll checks passed: python-dotenv forgives some mistakes, not all. Follow the strict rules.")

BASE_URL       -> 'https://api.groq.com/openai/v1'
MODEL          -> 'openai/gpt-oss-20b'
API_KEY        -> 'demo_key_one'
QUOTED         -> 'demo_key_two'
SPACES         -> 'demo_key_three'
SMART_QUOTES   -> '“demo_key_four”'
COMMENT_AFTER  -> 'openai/gpt-oss-20b'
HASH_NO_SPACE  -> 'openai/gpt-oss-20b# pinned'

All checks passed: python-dotenv forgives some mistakes, not all. Follow the strict rules.


**What you should see:** eight settings (the comment line is not one of them), then the "All checks passed" line. Look closely at two values: `SMART_QUOTES` still has its curly quotes `“ ”`, and `HASH_NO_SPACE` has `# pinned` glued to the model name. Each of those would give you a 401 (wrong key) or a 404 (model not found), with a `.env` that *looks* right.

**The take-away:** a `.env` file is plain text, read line by line. Curly quotes usually come from copying a key through Word, WhatsApp or an email; copy keys from the console straight into Notepad or VS Code instead.

### Under the hood 2: printing a key safely
Sometimes you do need to check *which* key is loaded (for example, after you created a new one). The safe way is to show only a few characters. This cell tests our `mask` function on a fake key.

**What the next cell does:**
1. Builds a **fake** key in code (`"gsk_" + "Demo" * 13`), so no real-looking key is ever saved in this notebook file.
2. Masks it and checks, with `assert`, that the full key does **not** appear in the masked text, while the start and end do.
3. Shows what happens with a short value (`ollama`, the dummy key Ollama uses) and with no value.
4. Prints your own key, masked. This line is safe to show on a projector.

In [7]:
# Under the hood 2 · printing a key safely   (fake key, built in code)
fake_key = "gsk_" + "Demo" * 13                 # NOT a real key
shown = mask(fake_key)
print("masked  :", shown)
assert fake_key not in shown                     # the full key never appears
assert shown.startswith(fake_key[:4]) and fake_key[-4:] in shown
print("short   :", mask("ollama"))               # short values are hidden completely
print("missing :", mask(None))
print("your key:", mask(API_KEY))                # safe to show; the full key is not

masked  : gsk_...Demo (56 characters)
short   : ******
missing : (missing)
your key: gsk_...pxVa (56 characters)


**What you should see:** `masked  : gsk_...Demo (56 characters)`, then `******`, then `(missing)`, then your own key masked (or `(missing)`).

### Check yourself
*Question 3 — Your `.env` contains `API_KEY=“gsk_...”` copied from a WhatsApp message to yourself. The file looks right, but the checkpoint fails with 401. Using Under the hood 1, explain why.*

Answer: Curly quotes or unwanted characters may become part of the key value. The provider may reject it and return a 401 Unauthorized error. Copy the actual key carefully and avoid adding extra quotation marks or spaces.

> **Common mistake:** keeping the key in a chat, an email draft or a screenshot "so I don't lose it". If you lose it, create a new one: it takes a minute. A key in a chat is a key you no longer control.

## 3. Block 2 · Make the client

### What and why
The **client** is the Python object that knows how to talk to a provider. You create it once with two of the three settings: `base_url` (where) and `api_key` (who). The third setting, `MODEL`, is given with each request, because one client can ask different models.

We also define one small helper, `explain`, that turns the classic errors into the fix from the README. You will use it in every model cell, so that an error message tells you *what to do*, not just what went wrong.

### The idea: the phone and the number
```
 OpenAI(base_url=BASE_URL, api_key=API_KEY)   →  client        (a phone, set up with a number and a PIN)
                                                  │             nothing has been sent yet
 client.chat.completions.create(...)          →  the call      (section 5: now a request travels)
```
**Analogy — the phone you dial with.** Buying and setting up a phone does not call anybody. Creating the client does not contact Groq, does not check your key and costs nothing. Errors about a wrong key appear only when you make the first call. That is why a wrong key "works" in this section and fails in section 5.

**Why the class is called `OpenAI` even though we use Groq:** the `openai` package speaks the **OpenAI-compatible** format, and Groq and Ollama both accept that format at their own `base_url`. The name of the class is the name of the format, not the company you are talking to.

<details><summary><b>▸ Read more: what happens if BASE_URL is missing?</b></summary>

If `base_url` is `None`, the `openai` package uses its default, OpenAI's own address (unless an `OPENAI_BASE_URL` variable is set). Your Groq key would then be sent to the wrong company and refused with a 401, which looks exactly like a bad key. If `api_key` is `None` (and there is no `OPENAI_API_KEY` variable), creating the client fails at once with an `OpenAIError` about a missing api_key. That is why this notebook only creates the client when all three settings are present (`ready`).
</details>

### What the next cell does
1. **`import openai`** gives us the error types (`openai.AuthenticationError` and friends); **`from openai import OpenAI`** gives us the client class.
2. **`client = OpenAI(...) if ready else None`** is a conditional expression again: make a client if the settings are complete, otherwise `None`.
3. Prints where the client points, or that it was not created.
4. **`def explain(error):`** checks the error's type with **`isinstance(error, SomeType)`** ("is this error a kind of SomeType?") and prints the matching fix: 401 bad key, 404 model not found, 429 rate limit, no connection. `str(error)[:200]` prints the first 200 characters of the provider's own message: a **slice**, `[:200]`, means "from the start up to position 200".

In [8]:
# Section 3 · Block 2 of hello.py: make the client   (nothing is sent in this cell)
import openai
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY) if ready else None      # creating a client sends NOTHING
print("client:", f"ready, pointing at {client.base_url}" if client else "not created (no key): model cells will say 'skipped'")


def explain(error):
    # Turn the classic API errors into the fix from the README.
    if isinstance(error, openai.AuthenticationError):
        print("401: the key was refused. Open .env (not .env.example): no quotes, no spaces, not the placeholder.")
    elif isinstance(error, openai.NotFoundError):
        print(f"404: model '{MODEL}' not found. Groq: run 'uv run python list_models.py' and tell the instructor "
              "(do not change MODEL yourself). Ollama: run 'ollama pull' for that model.")
    elif isinstance(error, openai.RateLimitError):
        print("429: free-tier rate limit. Wait about 30 seconds, then run the cell again.")
    elif isinstance(error, openai.APIConnectionError):
        print(f"No connection to {BASE_URL}. Check your internet; for Ollama, check that Ollama is running.")
    else:
        print("The call failed:", type(error).__name__)
    print("   provider says:", str(error)[:200])

client: ready, pointing at https://api.groq.com/openai/v1/


### What you should see
With a key: `client: ready, pointing at https://api.groq.com/openai/v1/`. (The trailing `/` is added by the library; it is normal.) Without a key: `client: not created (no key) ...`.

| If you see... | It means... | Do this |
|---|---|---|
| `OpenAIError: ... api_key ...` | you created a client yourself with no key | fix `.env`, restart the kernel, run from the top |
| it points at `api.openai.com` | `BASE_URL` was empty when the client was made | check the `BASE_URL=` line in `.env` |

### Check yourself
*Question 4 — You paste a wrong key into `.env`, restart, and run sections 2 and 3. Both run without any error. Why? In which section will you first see the problem, and with which error number?*

Answer: Creating the client does not necessarily send a request to the provider. The key is generally checked when the program makes an actual API request. An invalid key may then produce a 401 Unauthorized error.

> **Common mistake:** creating a new client inside every cell or every loop. One client is enough for the whole notebook; it is cheap to keep and it reuses its connection.

## 4. Block 3 · The messages: what you actually send

### What and why
A chat model does not receive "a question". It receives a **list of messages**, and each message has a **role** and some **content**. `hello.py` sends two:

- a **system** message: who the model is and how it should behave (*"You are a concise assistant for a retail bank."*);
- a **user** message: what was asked (*"Say hello and tell me one thing AI agents can do."*).

The reply comes back as a third kind of message, with the role **assistant**. In Session 19 you will send the whole conversation (user, assistant, user, assistant...) every time; the list just gets longer.

### The idea: the job description and the customer
```
 messages = [
   {"role": "system", "content": "You are a concise assistant for a retail bank."},    ◄── the job description
   {"role": "user",   "content": "Say hello and tell me one thing AI agents can do."}, ◄── the customer at the counter
 ]
                         │  sent with model=MODEL to BASE_URL + "/chat/completions"
                         ▼
 {"role": "assistant", "content": "Hello! One thing AI agents can do is ..."}           ◄── the reply
```
**Analogy.** A new employee at a bank branch gets a job description on day one (the system message) and then serves customers one by one (user messages). The job description shapes every answer, but the customer decides the question.

The list is plain Python: a **list** (`[...]`) of **dictionaries** (`{...}`), each with the keys `"role"` and `"content"`. JSON, the text format that travels over the internet, looks almost the same, which is why the conversion is automatic.

### What the next cell does
1. Builds `messages` exactly as in `hello.py`: a list of two dictionaries.
2. Loops over the list and prints each role and content. `m['role']` reads the value stored under the key `"role"`. In the f-string, `:>6` right-aligns the role in 6 characters so the colons line up.

In [9]:
# Section 4 · Block 3 of hello.py: the messages   (nothing is sent in this cell)
import json

messages = [
    {"role": "system", "content": "You are a concise assistant for a retail bank."},
    {"role": "user", "content": "Say hello and tell me one thing AI agents can do."},
]
for m in messages:
    print(f"{m['role']:>6}: {m['content']}")

system: You are a concise assistant for a retail bank.
  user: Say hello and tell me one thing AI agents can do.


### What you should see
```
system: You are a concise assistant for a retail bank.
  user: Say hello and tell me one thing AI agents can do.
```

### Under the hood 3: the request, as it travels
The client turns your Python call into an HTTPS request: an address, some headers, and a **body** written in JSON. You can build that body yourself and look at it. No key and no internet are needed, because nothing is sent.

**What the next cell does:**
1. Builds `request_body`, the same dictionary the client sends when you call `create(model=..., messages=..., stream=True, stream_options=...)`. `MODEL or "openai/gpt-oss-20b"` uses your model, or the course default from `.env.example` if you have no `.env` yet.
2. Prints the address the request goes to (`BASE_URL` + `/chat/completions`) and the header that carries the key, **masked**.
3. **`json.dumps(request_body, indent=2)`** turns the dictionary into JSON text, indented so you can read it.
4. Checks with `assert` that the key is **not** in the body: it travels only in the `Authorization` header.

In [10]:
# Under the hood 3 · the request body: what actually travels to BASE_URL + "/chat/completions"
request_body = {
    "model": MODEL or "openai/gpt-oss-20b",            # your MODEL, or the course default if .env is missing
    "messages": messages,
    "stream": True,
    "stream_options": {"include_usage": True},
}
print("POST", (BASE_URL or "https://api.groq.com/openai/v1") + "/chat/completions")
print("Authorization: Bearer", mask(API_KEY))           # the key goes in a header, never in the body
print(json.dumps(request_body, indent=2))
assert not API_KEY or API_KEY not in json.dumps(request_body)

POST https://api.groq.com/openai/v1/chat/completions
Authorization: Bearer gsk_...pxVa (56 characters)
{
  "model": "openai/gpt-oss-20b",
  "messages": [
    {
      "role": "system",
      "content": "You are a concise assistant for a retail bank."
    },
    {
      "role": "user",
      "content": "Say hello and tell me one thing AI agents can do."
    }
  ],
  "stream": true,
  "stream_options": {
    "include_usage": true
  }
}


**What you should see:** a `POST` line with the full address, a masked `Authorization` line, then the JSON body: `model`, the two messages, `"stream": true` and `"stream_options": {"include_usage": true}`. Notice that Python's `True` became JSON's `true`. That small block of text is *everything* the model receives from you. There is no hidden context about you, your laptop or your earlier calls: if it is not in `messages`, the model does not know it.

### Check yourself
*Question 5 — Predict first: if you delete the system message and keep only the user message, will the request still work? What do you expect to change in the answer?* (You can try it in section 7.)

Answer: The model may still generate a response, but its behavior can change because it no longer receives the system instructions.
For example, the system message might tell the model to respond as a helpful tutor. Removing it means that instruction is no longer supplied.

> **Common mistake:** expecting the model to remember the previous call. Each request is independent. To continue a conversation, you send the earlier messages again, in the list (Session 19).

## 5. Block 4a · The first call, all at once

### What and why
Now we send the messages. We do it first **without streaming**, because it is the easiest way to see the shape of the reply: you call, you wait, and you get one complete response object back. `hello.py` streams (section 6); both ways send the same request and get the same kind of answer.

### The idea: a letter, not a phone call
```
 client.chat.completions.create(model=MODEL, messages=messages)
        │   (you wait while the whole answer is written)
        ▼
 response
   ├─ .choices[0].message.content   ← the answer text
   ├─ .choices[0].finish_reason     ← why it stopped ("stop" = finished normally)
   └─ .usage                        ← the bill
        ├─ .prompt_tokens           what you sent (plus tokens the provider adds around your messages)
        ├─ .completion_tokens       what the model produced
        └─ .total_tokens            the two added up
```
**Analogy.** Non-streaming is a letter: nothing arrives until the whole letter is written, then you read it at once. Streaming (next section) is a phone call: you hear the words as they are spoken.

**Why `choices[0]`?** The API can return several alternative answers in one call (`choices` is a list). We ask for one, so we take the first: in Python, lists start at position `0`.

### What the next cell does
1. **`runs = []`** starts an empty list where we will keep each call's token counts, to compare them in section 7.
2. **`if not ready:`** prints `skipped (no key in .env)` and does nothing else. This is how every model cell starts.
3. Otherwise, inside **`try:`**, it sends the request with `create(model=MODEL, messages=messages)` and waits for the full reply.
4. Prints the answer text: `response.choices[0].message.content`.
5. Reads `response.usage` into `u` and prints the three counts in the same format as `hello.py`.
6. **`runs.append((...))`** adds a **tuple** (a fixed group of values in round brackets) to the list: a name and the three numbers.
7. **`except openai.APIError as e:`** catches any error from the API (401, 404, 429, no connection: all are kinds of `APIError`) and calls `explain(e)`, so you get the fix instead of a long red traceback.

In [11]:
# Section 5 · Block 4a: the first call, all at once   (1 call, a few hundred tokens)
runs = []                                                   # (name, prompt, completion, total) for each call
if not ready:
    print("skipped (no key in .env)")
else:
    try:
        response = client.chat.completions.create(model=MODEL, messages=messages)   # waits for the WHOLE answer
        print(response.choices[0].message.content)
        u = response.usage
        print(f"\nprompt={u.prompt_tokens} completion={u.completion_tokens} total={u.total_tokens}")
        runs.append(("section 5 · not streamed", u.prompt_tokens, u.completion_tokens, u.total_tokens))
    except openai.APIError as e:
        explain(e)

Hello!  
AI agents can automatically handle customer support inquiries, answering questions and resolving issues around the clock.

prompt=96 completion=89 total=185


### What you should see
A short greeting from the "retail bank assistant", then a token line. For example (from the deck: an illustration, not your run):
```
Hello! One thing AI agents can do is help you check your
account balance or recent transactions.

prompt=96 completion=105 total=201
```
**How to read it:** your text and your numbers *will* be different from this, from your neighbour's, and from your own next run. The model picks its words with some randomness, so the answer length changes, and the completion count changes with it. (Session 11 shows the setting, temperature, that controls this.)

Two surprises worth noticing:
- **The prompt count is bigger than your two short sentences.** The provider wraps your messages in some extra tokens of its own before the model sees them. Session 9b shows exactly what they are.
- **The completion count can be bigger than the visible answer.** The course's default model is a *reasoning model*: it writes some hidden working before the visible answer, and those tokens are counted (and billed) as completion tokens. Session 15 measures this.

| If you see... | It means... | Do this |
|---|---|---|
| `skipped (no key in .env)` | `ready` was `False` in section 2 | fix `.env`, **Kernel → Restart**, run from the top |
| `401: the key was refused` | wrong, incomplete or placeholder key; or quotes | see section 2's table; if in doubt, create a new key |
| `404: model ... not found` | the pinned model was retired, or `MODEL` was edited | `uv run python list_models.py` in the terminal; tell the instructor |
| `429: free-tier rate limit` | too many tokens in the last minute | wait about 30 seconds and run again |
| `No connection to ...` | no internet, a blocked network, or (Ollama) Ollama is not running | check Wi-Fi/hotspot; for Ollama see section 8 |

### Practice without a key: reading a response object
Whether or not you have a key, you can practise reading a response. The library has the same classes it uses for real replies; here we fill one with **made-up values** (an illustration, not a real run) and read it with exactly the same code as above.

**What the next cell does:**
1. **`ChatCompletion.model_validate({...})`** builds a response object from a dictionary shaped like the JSON a provider sends back.
2. Reads the text, the finish reason and the usage the same way as the real call.

In [21]:
# Practice without a key · reading a response object   (MADE-UP reply: an illustration, not a real run)
from openai.types.chat import ChatCompletion

practice = ChatCompletion.model_validate({
    "id": "practice-1", "object": "chat.completion", "created": 0, "model": "practice-model",
    "choices": [{"index": 0, "finish_reason": "stop",
                 "message": {"role": "assistant",
                             "content": "Hello! An AI agent can check a loan application for missing documents."}}],
    "usage": {"prompt_tokens": 40, "completion_tokens": 15, "total_tokens": 55},
})
print("text         :", practice.choices[0].message.content)
print("role         :", practice.choices[0].message.role)
print("finish_reason:", practice.choices[0].finish_reason)
u = practice.usage
print(f"usage        : prompt={u.prompt_tokens} completion={u.completion_tokens} total={u.total_tokens}")

text         : Hello! An AI agent can check a loan application for missing documents.
role         : assistant
finish_reason: stop
usage        : prompt=40 completion=15 total=55


**What you should see:** the made-up text, `role : assistant`, `finish_reason : stop`, and `prompt=40 completion=15 total=55` (made-up numbers). The point: `response.choices[0].message.content` and `response.usage` are the two paths you will type most often in this course.

### Check yourself
*Question 6 — Run the section 5 cell twice. What changed between the two runs, and what did not? Why can the completion count change when the question did not?*
Answer:

When the Section 5 cell is run twice with the same question, the model may generate different responses.

- What changed: The completion count (the number of tokens generated in the response) may change because the model can produce a response with different wording, length, or detail.

- What did not change: The question and the input prompt remain the same, assuming no changes were made to the cell or its settings.

- Why can the completion count change? The model generates responses probabilistically, so the same question can produce different answers. A longer response uses more completion tokens, while a shorter response uses fewer.

In simple words: The question stays the same, but the AI may answer differently each time. Because the answer length can change, the completion token count can also change.

> **Common mistake:** printing `response` itself and trying to read the answer from that long block. Go straight to `response.choices[0].message.content` for the text and `response.usage` for the bill.

## 6. Block 4b · The same call, streamed (exactly like `hello.py`)

### What and why
`hello.py` asks for the answer with **`stream=True`**. The reply then arrives in many small pieces, **chunks**, while the model is still writing. You print each piece as soon as it arrives, so the first words appear almost at once. Every chat product works this way, because waiting several seconds for a blank screen feels broken.

Streaming changes one thing about the bill: the usage is not on a response object any more, because there is no single response object. With **`stream_options={"include_usage": True}`** the provider sends **one extra chunk at the very end** that carries the usage. Without that option, a streamed reply usually has no token counts at all.

### The idea: a live scoreboard
```
 chunk 1   choices[0].delta.content = ""         (often empty: just says "the assistant starts")
 chunk 2   choices[0].delta.content = "Hello"
 chunk 3   choices[0].delta.content = "! One"
  ...      ...                                   print each piece with end="" → one flowing line
 chunk N   choices[0].delta.content = "."
 last      choices = []    usage = {prompt, completion, total}   ◄── only because include_usage=True
```
**Analogy — the live cricket score.** You see each ball as it happens (each chunk), and at the end of the innings you get the final scorecard (the usage chunk).

Each chunk has a **`delta`** (the *new* piece of text) instead of a full `message`. That is why the loop prints `delta.content`, not `message.content`.

### What the next cell does
This is `hello.py`'s Block 3 and Block 4, with two small additions (a chunk counter, and error handling).
1. **`create(..., stream=True, stream_options={"include_usage": True})`** returns a **stream**: something you loop over, chunk by chunk, as the chunks arrive.
2. **`for chunk in stream:`** runs once per chunk. `chunks += 1` counts them (`+=` means "add to the current value").
3. **`if chunk.choices and chunk.choices[0].delta.content:`** has two checks joined by `and`. First: does this chunk have any choices? (The final usage chunk has an empty list, and `chunk.choices[0]` on an empty list would crash.) Second: is there text in it? (The first chunk often has an empty string.) Python reads an empty list, an empty string and `None` as *false*, so the `if` skips those chunks.
4. **`print(..., end="", flush=True)`**: `end=""` means "do not start a new line after this piece", so the pieces join up; `flush=True` means "show it now", not when Python's output buffer is full.
5. **`if chunk.usage:`** is true only on the final chunk: it prints the three counts, as in `hello.py`, and adds them to `runs`.
6. `except openai.APIError as e: explain(e)` as before.

In [13]:
# Section 6 · Block 4b: the same call, streamed, like hello.py   (1 call)
if not ready:
    print("skipped (no key in .env)")
else:
    try:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            stream=True,                                   # pieces arrive while the model writes
            stream_options={"include_usage": True},        # ...and one last chunk carries the token counts
        )
        chunks = 0
        for chunk in stream:
            chunks += 1
            if chunk.choices and chunk.choices[0].delta.content:
                print(chunk.choices[0].delta.content, end="", flush=True)
            if chunk.usage:
                u = chunk.usage
                print(f"\n\nprompt={u.prompt_tokens} completion={u.completion_tokens} total={u.total_tokens}")
                runs.append(("section 6 · streamed", u.prompt_tokens, u.completion_tokens, u.total_tokens))
        print(f"({chunks} chunks arrived)")
    except openai.APIError as e:
        explain(e)

Hello! One thing AI agents can do is help customers find the best loan rates quickly.

prompt=96 completion=86 total=182


prompt=96 completion=86 total=182
(79 chunks arrived)


### What you should see
The answer appears **piece by piece** (watch closely; with a fast provider it can be quick), then the token line, then the number of chunks. This is exactly what `uv run python hello.py` prints in the terminal, plus the chunk count.

| If you see... | It means... | Do this |
|---|---|---|
| the answer, but no `prompt=...` line | the usage chunk did not arrive (some providers or old servers ignore `include_usage`) | check you kept `stream_options={"include_usage": True}`; on Ollama, make sure it is up to date |
| every piece on its own line | `end=""` was removed | put `end=""` back in the `print` |
| `IndexError: list index out of range` | the `chunk.choices and` check was removed | put it back: the last chunk has no choices |
| an error from `explain` | same causes as section 5 | same fixes as section 5 |

### Practice without a key: reading a stream
Here is a **made-up** stream (an illustration, not a real run) built from the library's own chunk class, read by **exactly the same loop**. It shows the empty first chunk and the final usage chunk with no choices.

**What the next cell does:**
1. `fake_chunk(text, usage)` builds one chunk: with text it has one choice; with `text=None` it has an empty `choices` list (like the real final chunk).
2. **`[fake_chunk(piece) for piece in [...]]`** is a **list comprehension**: a one-line way to build a list by applying something to each item. Here it turns each piece of text into a chunk. Then we add (`+`) one final usage chunk.
3. The `for chunk in fake_stream:` loop is character for character the loop from `hello.py`.

In [14]:
# Practice without a key · reading a stream   (MADE-UP chunks: an illustration, not a real run)
from openai.types.chat import ChatCompletionChunk


def fake_chunk(text=None, usage=None):
    choices = [] if text is None else [{"index": 0, "delta": {"content": text}, "finish_reason": None}]
    return ChatCompletionChunk.model_validate({"id": "practice", "object": "chat.completion.chunk", "created": 0,
                                               "model": "practice-model", "choices": choices, "usage": usage})


pieces = ["", "Hello", "! An", " AI", " agent", " can", " check", " a loan", " file", "."]
fake_stream = [fake_chunk(piece) for piece in pieces] + [
    fake_chunk(usage={"prompt_tokens": 40, "completion_tokens": 12, "total_tokens": 52})]

for chunk in fake_stream:                                       # the same loop as hello.py
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.usage:
        u = chunk.usage
        print(f"\n\nprompt={u.prompt_tokens} completion={u.completion_tokens} total={u.total_tokens}")
print(f"({len(fake_stream)} chunks: the first had empty text, the last had no choices, only usage)")

Hello! An AI agent can check a loan file.

prompt=40 completion=12 total=52
(11 chunks: the first had empty text, the last had no choices, only usage)


**What you should see:** `Hello! An AI agent can check a loan file.` on one line, then `prompt=40 completion=12 total=52` (made-up numbers), then `(11 chunks: ...)`. Try deleting `chunk.choices and` from the `if` and re-run: you will get the `IndexError` from the table above. Put it back.

### Check yourself
*Question 7 — Streaming and non-streaming sent the same messages. Which one shows you the first word sooner? Which one finishes sooner? Why do you need `include_usage` only when streaming?*

Answers:

1. Which one shows the first word sooner?

Streaming shows the first word sooner because it displays the response in small chunks as the model generates them, instead of waiting for the complete answer.

2. Which one finishes sooner?

Non-streaming may finish sooner in terms of total request time, depending on the model and network. Streaming can feel faster because you see the response while it is being generated, but it does not necessarily reduce the total generation time.

3. Why do you need include_usage only when streaming?

In a non-streaming request, token usage information is normally included in the final response. In a streaming request, the response arrives in chunks, so usage information may not be included by default. Setting stream_options={"include_usage": True} requests token usage information in the final streaming chunk, if the API supports it.

In simple words: Streaming lets you see the answer sooner, while non-streaming waits for the complete answer. include_usage helps you obtain token counts when using streaming.

> **Common mistake:** thinking streaming makes the model faster or cheaper. The total time and the tokens are about the same; you just see the start sooner. And without `include_usage`, a streamed call can be "free" only in the sense that you did not *see* the bill.

## 7. Read the bill: change the question, compare the tokens

### What and why
In the deck this is "the number that matters". Every call has a price, measured in tokens, and it is printed at the bottom of your screen. In week eight you will build an agent, and an agent does not make one call: it makes five, ten, fifty, in a loop. The token line is the number that runs away if nothing stops it. Today you just learn to read it, and to see what makes it move.

In class you changed the question inside `hello.py`. Here you change it in a cell instead, which has one advantage: `hello.py` stays exactly as the course repository has it, so `git pull` never complains about your local changes.

### The idea: same code, different data
```
 messages[0]  system: "You are a concise assistant for a retail bank."     (kept)
 my_messages  user:   my_question                                          (YOU change this)
                          │
                          ▼  the same create(...) and the same loop
 different answer  →  different completion tokens  →  different total
 a longer question →  more prompt tokens
```
The question is **data**; the mechanics are **code**. The code did not change, but the bill did.

### What the next cell does
1. **`my_question = "Explain EMI in one sentence."`**: change this text, then run the cell again. (EMI, equated monthly instalment, is the fixed monthly loan payment.)
2. **`my_messages = [messages[0], {...}]`** builds a new list: the same system message (`messages[0]`, the first item) and a new user message with your question.
3. Sends it streamed, exactly like section 6, and adds the counts to `runs`.
4. Prints a small table of **every run so far** in this notebook. **`for name, p, c, t in runs:`** unpacks each tuple into four variables in one go. In the f-strings, `:<26` pads to the left and `:>8` aligns numbers to the right.

In [15]:
# Section 7 · Read the bill   (1 call each time you run it)
my_question = "Explain EMI in one sentence."             # YOUR TURN: change the question, then run again
my_messages = [messages[0], {"role": "user", "content": my_question}]    # same system message, new user message

if not ready:
    print("skipped (no key in .env)")
else:
    try:
        stream = client.chat.completions.create(model=MODEL, messages=my_messages, stream=True,
                                                stream_options={"include_usage": True})
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                print(chunk.choices[0].delta.content, end="", flush=True)
            if chunk.usage:
                u = chunk.usage
                runs.append((f"section 7 · {my_question[:14]}", u.prompt_tokens, u.completion_tokens, u.total_tokens))
        print("\n")
        print(f"{'run':<26}{'prompt':>8}{'completion':>12}{'total':>8}")
        for name, p, c, t in runs:
            print(f"{name:<26}{p:>8}{c:>12}{t:>8}")
    except openai.APIError as e:
        explain(e)

EMI, or Equated Monthly Installment, is the fixed monthly payment you make to repay a loan over a set period.

run                         prompt  completion   total
section 5 · not streamed        96          89     185
section 6 · streamed            96          86     182
section 6 · streamed            96          86     182
section 7 · Explain EMI in      90          86     176
section 7 · Explain EMI in      90          86     176


### What you should see
A one-sentence explanation of EMI, then a table with one row per call you have made in this notebook (an illustration, not a real run; your numbers will differ):
```
run                         prompt  completion   total
section 5 · not streamed        96         110     206
section 6 · streamed            96          98     194
section 7 · Explain EMI in      90          75     165
```
**How to read it:**
- Sections 5 and 6 sent the **same messages**, so their prompt counts should be equal. Their completion counts usually differ, because the answers differ.
- Your own question changes the prompt count a little (a different number of words) and the completion count a lot (a different answer).
- Run the cell several times with different questions: a long question raises the prompt count; asking for "three examples" instead of "one sentence" raises the completion count.

If the table has only one row, sections 5 and 6 were skipped or failed. That is fine; the row you have is still real.

### Check yourself
*Question 8 — Ask for "one sentence", then for "five bullet points" on the same topic. Which count changes most, prompt or completion? What does that tell you about controlling cost through the wording of a request?*

Answers:

1. Which count changes most?

The completion token count usually changes the most because five bullet points generally require a longer response than one sentence. The prompt token count may change slightly if the wording of the request changes.

2. Why does this happen?

The prompt contains the instructions sent to the model, while completion tokens represent the answer generated by the model. Asking for five bullet points encourages the model to generate more output than asking for one sentence.

3. What does this tell us about controlling cost?

We can help control API costs by giving clear instructions about the required answer length and format. Shorter answers generally use fewer completion tokens and may cost less.

In simple words: Asking for one sentence usually uses fewer output tokens than asking for five bullet points. Clear instructions such as “answer in one sentence” or “keep it under 50 words” can help reduce token usage and control costs.

> **Common mistake:** comparing your numbers with your neighbour's and thinking one of you is wrong. Different answers mean different completion counts. Compare *your* runs with each other, changing one thing at a time.

## 8. One code, two providers: Groq or Ollama

### What and why
The course uses **Groq** by default: a hosted provider with a free tier and a fast model. The fallback is **Ollama**: a program that runs a small model on your own laptop. You switch between them by changing `.env` only. Nothing in `hello.py` or this notebook changes.

Why have a fallback at all? Two reasons you will meet for real:
1. **Rate limits.** On the free tier, if the whole class sends requests at once, some get a 429. A local model has no rate limit.
2. **Private data.** Some data must not leave the building (a hospital's notes, a bank's customer records). A local model keeps everything on the machine. Session 12 goes deeper.

### The idea: the same card, a different ATM
```
 .env for Groq (default)                         .env for Ollama (local)
 ───────────────────────────────────────         ──────────────────────────────────────────
 BASE_URL=https://api.groq.com/openai/v1         BASE_URL=http://localhost:11434/v1
 API_KEY=<your Groq key>                         API_KEY=ollama        (any text; Ollama ignores it)
 MODEL=openai/gpt-oss-20b                        MODEL=llama3.2:3b
            │                                                │
            └──────────── the SAME client, the SAME code ────┘
```
**How to switch (from the README):** in `.env`, put `#` in front of the three Groq lines, remove the `#` from the three Ollama lines, save, and **restart the kernel**. To switch back, do the opposite. Before the first switch, install Ollama from ollama.com and run `ollama pull llama3.2:3b` (about 2 GB: do it at home on good Wi-Fi).

**Analogy.** Your card works at different banks' ATMs because they all accept the same format. The OpenAI-compatible format is that card standard; Groq and Ollama are two ATMs.

### What the next cell does
1. Works out the provider from `BASE_URL` with a chained conditional expression: `"Groq" if ... else "Ollama ..." if ... else ...` reads left to right and picks the first true condition. `(BASE_URL or "")` turns `None` into an empty string, so `in` cannot crash.
2. If `ready`, asks the provider for the list of models your key can use: **`client.models.list()`**, exactly like `list_models.py`. This costs **no tokens**: it is a list, not a generation.
3. **`sorted(m.id for m in client.models.list())`** takes the `id` of each model and sorts the names alphabetically. `m.id for m in ...` is a **generator expression**: like a list comprehension, without the square brackets, when the result goes straight into a function.
4. Prints the first ten names (`names[:10]`, a slice), and whether your pinned `MODEL` is in the list. `MODEL in names` is `True` or `False`.

In [16]:
# Section 8 · Which provider does your .env point at, and which models can your key use?   (no tokens)
provider = ("Groq" if "groq.com" in (BASE_URL or "")
            else "Ollama (on your laptop)" if "11434" in (BASE_URL or "")
            else "another provider" if BASE_URL else "none (no .env)")
print("your .env points at:", provider)

if not ready:
    print("skipped (no key in .env)")
else:
    try:
        names = sorted(m.id for m in client.models.list())     # the same as list_models.py; costs no tokens
        print(len(names), "models visible, for example:")
        for name in names[:10]:
            print("   ", name)
        print("your pinned MODEL is in the list:", MODEL in names)
    except openai.APIError as e:
        explain(e)

your .env points at: Groq
11 models visible, for example:
    allam-2-7b
    canopylabs/orpheus-arabic-saudi
    canopylabs/orpheus-v1-english
    meta-llama/llama-prompt-guard-2-22m
    meta-llama/llama-prompt-guard-2-86m
    openai/gpt-oss-120b
    openai/gpt-oss-20b
    openai/gpt-oss-safeguard-20b
    qwen/qwen3.8-27b
    whisper-large-v3
your pinned MODEL is in the list: True


### What you should see
`your .env points at: Groq`, then a number of models and the first ten names in alphabetical order, then `your pinned MODEL is in the list: True`. The exact list changes over time, as providers add and retire models; that is normal, and it is exactly why `MODEL` is pinned and checked.

With Ollama: the provider line says Ollama, and the list shows only the models you have pulled (for example `llama3.2:3b`).

| If you see... | It means... | Do this |
|---|---|---|
| `pinned MODEL is in the list: False` | the pinned model was retired (expect 404s) | tell the instructor; do not change `MODEL` yourself |
| Ollama: `No connection to http://localhost:11434/v1` | Ollama is not running | check the tray icon, or run `ollama serve` in a second terminal |
| Ollama: 404 model not found when calling | the model was never pulled | `ollama pull llama3.2:3b` |
| Ollama answers are slow | a small laptop runs the model on its CPU | normal; Session 12 measures it |

### Check yourself
*Question 9 — Your manager says: "For the pilot with real customer data, the model must run on our own machine." Using only this session, describe exactly what you would change, and what you would not change.*

Answer:

To run the model on our own machine, I would use a locally running model through Ollama and change the environment configuration in the .env file.

1. What I would change

I would update the .env file with the following settings:

BASE_URL=http://localhost:11434/v1

API_KEY=ollama

MODEL=llama3.2:3b


- BASE_URL: Changes the API endpoint to the local Ollama server.

- API_KEY: Uses the value expected by this local setup.

- MODEL: Specifies the local model, llama3.2:3b.

I would also ensure that Ollama is running and that the specified model is available locally. After changing the configuration, I would restart the notebook kernel if necessary.

2. What I would not change

- I would not change the application code if the local server supports the same OpenAI-compatible API.

- I would keep the existing message structure, including the system and user messages.

- I would keep the existing request logic for sending prompts and receiving responses.

3. Why would I do this?

The application can use a locally running model by changing its configuration rather than rewriting the entire program. This allows the pilot to send requests to the local model instead of the hosted API.

In simple words: Change the BASE_URL, API_KEY, and MODEL values in .env, make sure Ollama and the model are running locally, and keep the application code unchanged if the API is compatible.

> **Common mistake:** switching `.env` to Ollama and forgetting to restart the kernel, then wondering why the answer still comes from Groq. The notebook read `.env` once, in section 2. Restart, run from the top.

## 9. The checkpoint

### What and why
Every session in this course ends the same way: a test goes green, or you are not finished. Today's test is `tests/test_setup.py`. It has three tests, and each one checks something this notebook explained:

| Test | What it checks | Why it fails, usually |
|---|---|---|
| `test_python_version` | the Python running the test is 3.12 | you typed `pytest ...` or `python -m pytest ...` without `uv run` |
| `test_env_present` | `BASE_URL`, `API_KEY` and `MODEL` exist; the key is not the placeholder and does not start with a quote | `.env` missing, placeholder not replaced, quotes around the key |
| `test_model_answers` | one tiny request ("Reply with OK", at most 64 tokens) gets an answer | wrong key (401), retired model (404), no internet |

The test file starts with the same line as `hello.py`, `load_dotenv(override=True)`, so it reads your `.env` the same way.

### The idea: the examiner
**Analogy.** The test is the examiner: it does not care how you got there, only whether the result is right. *"It ran without errors"* is not the same as *"it passed"*, and **skipped is never passed**.

### What the next cell does
1. If you have no key, it prints `skipped` and tells you what the terminal run would report.
2. Otherwise **`subprocess.run([...])`** runs another program from Python: here `python -m pytest tests/test_setup.py`, using **this notebook's own Python** (`sys.executable`, the `.venv` one if section 1 said yes) and the course folder as the working folder (`cwd=REPO`).
3. `capture_output=True, text=True` collects what pytest prints, as text, so the cell can show the last part of it.

The terminal command is still the official checkpoint; this cell is a convenience.

In [18]:
# Section 9 · The checkpoint, run from the notebook   (1 tiny call inside the test; the terminal run is the official one)
import subprocess

if not ready:
    print("skipped (no key in .env). In the terminal, 'uv run pytest tests/test_setup.py' would fail test_env_present.")
else:
    result = subprocess.run([sys.executable, "-m", "pytest", "tests/test_setup.py"],
                            cwd=REPO, capture_output=True, text=True)
    print((result.stdout or result.stderr)[-1500:])

...                                                                      [100%]
3 passed in 16.95s



### What you should see
With a key, the end of pytest's output, finishing with:
```
3 passed
```
(possibly with a time, like `3 passed in 1.23s`). Now do the official run in the VS Code terminal (Ctrl+`), in the course folder, and show it to the lab assistant:

```powershell
uv run pytest tests/test_setup.py
```

| If you see... | It means... | Do this |
|---|---|---|
| `test_python_version` FAILED | not the project's Python | in the terminal, always `uv run pytest ...`; in the notebook, fix the kernel (section 1) |
| `test_env_present` FAILED with a message | the message says exactly what is wrong (missing, placeholder, quotes) | fix `.env` as the message says |
| `test_model_answers` FAILED with `AuthenticationError` | 401: the key was refused | create a new key in the console, paste it again |
| `test_model_answers` FAILED with `NotFoundError` | 404: the model was retired | `uv run python list_models.py`; tell the instructor |
| `no tests ran` / `file not found` | pytest ran in the wrong folder | `cd ~\Documents\genai-agents-course` first |

### Check yourself
*Question 10 — Which of the three tests would still pass for a student with no key at all? Why is "two passed, one failed" still "not finished"?*

Answer:

1. Which tests would still pass without an API key?

Based on the session, the three tests check the Python version, environment settings, and whether the model can answer a question.

- Python version test: Can pass without an API key because it checks the Python environment.

- Environment settings test: May pass if the required settings are configured correctly, even if the API key is missing or invalid.

- Model response test: Would generally fail without a valid API key when using a hosted API, because the request cannot be authenticated.

The exact result depends on how the tests are implemented and whether they require a hosted API key.

2. Why is “two passed, one failed” still “not finished”?

The project checkpoint requires all three tests to pass. If one test fails, the setup is not fully verified, even if the other two pass.

In simple words: Two successful tests do not prove that the entire setup works. You must fix the failed test and rerun the tests until the checkpoint reports 3 passed.

> **Common mistake:** running `pytest` (or `python -m pytest`) without `uv run` and getting a red `test_python_version`. The test is doing its job: it caught the wrong Python.

## 10. Reflection

Answer in two or three sentences each, in your own words. These are the questions you will be asked again in Session 9b and in interviews.

1. In one sentence each: what do `uv sync` and `uv run` guarantee?
2. Why is `.env` git-ignored, and what would you do in the first five minutes after a key leaked?
3. Write down your three token lines from section 7. Which count moved most when you changed the question, and why?
4. You switched (or will switch) to Ollama tonight. Which file did you change, and which files did you *not* change?
5. What is one thing about the reply that surprised you (the prompt count, the speed of streaming, the answer itself)?

**Before Session 9b** (from the deck):
- `uv run pytest tests/test_setup.py` green on your own laptop, on your own Wi-Fi.
- Install Ollama (ollama.com) and run `ollama pull llama3.2:3b` (about 2 GB). Switch the three lines in `.env`, run `uv run python hello.py` once locally, switch back.
- Watch the Git & GitHub webinar this weekend. By Session 14 you will need your own repository for submissions.
- Read `module01/reading_tokens.md` (10 minutes).

---
# Part C · Wrap-up

## C1. Summary

- **One repo, one Python, one config file, one rule.** The course repository is pulled, never edited; `uv` gives the project its own Python 3.12 and exact packages in `.venv`; `.env` holds your settings; every Python command starts with `uv run`.
- **`uv sync` + `uv.lock` = the same environment on every laptop**, down to the patch version. That is what ends "it works on my laptop".
- **Three settings, no secrets in code.** `BASE_URL` (where), `API_KEY` (who), `MODEL` (which). `load_dotenv(override=True)` reads them from `.env`, which `.gitignore` keeps off GitHub. Print keys only masked.
- **`hello.py` in four blocks:** load the settings, make the client (sends nothing), build the messages (system + user), read the reply. The client is OpenAI-compatible, so Groq and Ollama work with the same code.
- **Two ways to read a reply:** all at once (`response.choices[0].message.content`, `response.usage`) or streamed (print each `delta.content`; the last chunk carries the usage only with `include_usage`).
- **Every call has a bill:** prompt, completion and total tokens. The question is data; changing it changes the bill, not the code.
- **Errors are messages, not disasters:** 401 = key, 404 = model, 429 = wait, no connection = network or Ollama not running.

## C2. The words again

| Term | One-line meaning |
|---|---|
| Course repo / work repo | The instructor's repository you pull; your own repository you push (from Session 14) |
| uv | Installs Python, creates the environment and installs pinned packages |
| `.venv` | The project's private Python and packages; never edited, never committed |
| `pyproject.toml` / `uv.lock` | The shopping list / the exact receipt of package versions |
| `uv sync` / `uv run` | Make `.venv` match the lock file / run a command with the project's Python |
| Environment variable | A named setting outside the code, read with `os.getenv` |
| `.env` / `.env.example` | Your private settings file (git-ignored) / the public template |
| `.gitignore` | Files Git never tracks, such as `.env` |
| API key | Your secret credential; like an ATM PIN; revoke and replace if leaked |
| `BASE_URL`, `API_KEY`, `MODEL` | Where the request goes, who is asking, which model answers |
| OpenAI-compatible | Accepts the same request format, so one client talks to many providers |
| Client | The object that sends requests; creating it sends nothing |
| Messages and roles | The request is a list of `{role, content}`: system, user (and assistant replies) |
| Streaming / chunk | The reply arrives in pieces while it is written; each piece is a chunk |
| Usage (prompt / completion / total) | The token bill for one call: what you sent, what came back, both together |
| Checkpoint test | `uv run pytest tests/test_setup.py` → `3 passed`; skipped is not passed |

## C3. Interview practice

Each answer has three parts: a definition, an example from this lab, and one honest limit.

**1. "Where do you keep API keys, and what do you do if one leaks?"**
*Definition:* keys are configuration, not code. Locally they live in a git-ignored file such as `.env` and are read as environment variables; in production they come from a secrets manager and reach the program the same way. *Example:* in this lab `load_dotenv(override=True)` reads `.env`, the code calls `os.getenv("API_KEY")`, `.gitignore` lists `.env`, and we print the key only masked. If one leaks, revoke it in the provider's console at once, create a new one, then find and remove the leak. *Limit:* `.gitignore` only protects against `git add`; it does not stop you pasting the key into chat or a screenshot, and once a key is pushed, deleting the commit is not enough: it must be revoked.

**2. "How do you make a Python project reproducible on another machine?"**
*Definition:* give the project its own environment with an exact, locked list of package versions, and always run through that environment. *Example:* `pyproject.toml` lists what we need, `uv.lock` pins exact versions, `uv sync` builds `.venv`, and `uv run` guarantees the project's Python 3.12; `test_python_version` catches anyone who forgot. *Limit:* the lock file pins Python packages, not everything: the operating system, a remote model that a provider retires, or a model's random sampling can still make two runs differ.

**3. "What does 'OpenAI-compatible API' mean? How would you switch from a hosted model to a local one?"**
*Definition:* the server accepts the same Chat Completions request format as OpenAI's API, so the same client library works with it by changing the base URL, key and model. *Example:* we switch from Groq to Ollama by changing three lines in `.env` (`http://localhost:11434/v1`, `ollama`, `llama3.2:3b`), with no change to `hello.py`. *Limit:* compatible does not mean identical: features such as `stream_options`, tool calling or the model list can differ between providers, and a small local model gives different quality and speed, so you still re-test after switching.

**4. "What are the system and user roles in a chat request?"**
*Definition:* the request is a list of messages; the system message sets who the model is and how it behaves, the user message is what was asked, and the model's reply has the assistant role. *Example:* our system message was "You are a concise assistant for a retail bank." and the user message asked for one thing AI agents can do; in section 7 we kept the system message and changed only the user message. *Limit:* the system message strongly guides the model but does not *force* it; a determined user message can still pull it off course (prompt injection, Sessions 18 and 55), and the model remembers nothing between requests unless you send the history again.

**5. "Why would you stream a response, and how do you still get the token count?"**
*Definition:* streaming sends the reply in chunks while it is generated, so users see the first words quickly; the token usage is sent in one final chunk when you ask for it. *Example:* `stream=True` with `stream_options={"include_usage": True}`; our loop prints each `delta.content` with `end=""` and reads `chunk.usage` from the last chunk, which has no choices (hence the `chunk.choices and` check). *Limit:* streaming does not make the whole answer faster or cheaper, it makes the *start* visible sooner; and not every provider or server version supports `include_usage`, so production code should handle a missing usage chunk.

## C4. Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `'uv' is not recognized` | the terminal was open before uv was installed | close **every** terminal (including VS Code's) and open a new one; still failing → README step 1b (adds `~\.local\bin` to PATH), then reopen |
| `'git' is not recognized` | same: terminal not reopened after installing Git | reopen the terminal |
| execution-policy error on the uv install line | a locked-down (company) laptop | run PowerShell as administrator once; if still blocked, pair with a neighbour today |
| `uv version` errors outside a project | newer uv uses `uv version` for the project's version | use `uv self version` |
| a warning that `UV_NATIVE_TLS` is deprecated | an old setting on some machines | harmless; ignore |
| `uv sync` hangs or fails on the network | classroom Wi-Fi | use a phone hotspot for this one step |
| the prompt says `C:\WINDOWS\system32>` | an administrator PowerShell opens there | `cd ~\Documents\genai-agents-course`; never clone into system32 |
| VS Code shows your whole drive | `code .` was run outside the project folder | *File → Open Folder* → `Documents\genai-agents-course` |
| `(base)` in the prompt; `python` runs Anaconda | Anaconda switches itself on | expected; use `uv run`; do not deactivate it |
| `No module named 'dotenv'` / `'openai'` | plain `python`, or Jupyter/VS Code using the wrong Python | `uv run ...`; in VS Code select the course `.venv` kernel; `uv sync` if needed |
| `.env found at : NOT FOUND` | no `.env`, wrong folder, or saved as `.env.txt` | `copy .env.example .env` in the course folder; turn on file-name extensions to check |
| `test_python_version` fails | the test ran on a Python that is not 3.12 | `uv run pytest tests/test_setup.py` |
| **401** invalid API key | quotes, trailing space, placeholder, wrong file (`.env.example`), curly quotes, or an incomplete copy | open `.env`; fix the line; if in doubt create a new key in the console and paste again |
| **404** model not found (Groq) | the pinned model was retired, or `MODEL` was edited | `uv run python list_models.py`; tell the instructor; do not change `MODEL` yourself |
| **404** and the model name ends with `# pinned...` | an old `.env` copied from a version of `.env.example` with a comment after the value | delete the comment from the `MODEL` line in `.env` |
| Ollama: model not found | the model was never pulled | `ollama pull llama3.2:3b` |
| Ollama: connection refused / no connection | Ollama is not running | check the tray icon, or `ollama serve` in a second terminal |
| **429** rate limit | the free tier's per-minute limit | wait about 30 seconds and run again; if it keeps happening, switch `.env` to Ollama |
| changed `.env`, nothing changed | the kernel read `.env` before your edit | **Kernel → Restart**, run from the top |
| `git pull` says your local changes to `hello.py` would be overwritten | you edited `hello.py` to change the question | `git restore hello.py`, then `git pull`; change questions in section 7 instead |

## C5. What comes next

**Session 9b · How LLMs work.** You saw today that `prompt=` was bigger than your two short sentences, and that the count changed with every answer. Session 9b opens that box: what a token really is, why some languages cost several times more tokens than English for the same sentence, how to estimate the cost of a call before you send it, and why the context window is a budget.

**Keep from today:**
- a green `uv run pytest tests/test_setup.py` on your own laptop;
- your `.env` (never committed), and the habit of restarting the kernel after you edit it;
- your three token lines from section 7, to compare with Session 9b's estimates;
- Ollama installed with `llama3.2:3b` pulled, so the fallback is ready when you need it (Sessions 11 and 12).